# 13 Boundary Duplication Study

Compare regular payoff encoding and duplicated payoff encoding for boundary sensitivity.


In [ ]:
import json
import math
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%run 00_project_setup_and_shared_functions.ipynb

config = load_config()
rng = set_global_seed(int(config["random_seed"]))
print("Config loaded and deterministic seed set.")


In [ ]:
spot = 24000.0; K = 24000.0; T = 30/365; r = 0.065; sigma = 0.18; option_type = "put"
rows = []
for n in [5, 6, 7, 8]:
    for duplicate in [False, True]:
        q = quantum_price_reconstruction(spot, K, T, r, sigma, option_type, n_qubits=n, x_width=float(config["quantum"]["x_width"]), use_duplication=duplicate)
        metrics = price_error_metrics(q["classical_curve"], q["price_curve"], q["S_grid"], K, option_type)
        edge_error = float(np.mean(np.abs((q["price_curve"] - q["classical_curve"])[[0, 1, -2, -1]])))
        rows.append({"n_qubits": n, "duplicated": duplicate, "post_selection_probability": q["post_selection_probability"], "edge_error": edge_error, **metrics})
study = pd.DataFrame(rows)
save_table(study, "13_boundary_duplication_study.csv")
plt.figure()
for duplicate, group in study.groupby("duplicated"):
    plt.plot(group["n_qubits"], group["RMSE"], marker="o", label=f"duplicated={duplicate}")
plt.title("Boundary duplication comparison")
plt.xlabel("Grid qubits")
plt.ylabel("RMSE")
plt.legend()
save_current_figure("13_boundary_duplication_rmse.png")
plt.figure()
for duplicate, group in study.groupby("duplicated"):
    plt.plot(group["n_qubits"], group["edge_error"], marker="o", label=f"duplicated={duplicate}")
plt.title("Boundary edge error comparison")
plt.xlabel("Grid qubits")
plt.ylabel("Edge MAE")
plt.legend()
save_current_figure("13_boundary_duplication_edge_error.png")
study
